# Telco Dataset Integration

## 1. Orchestration boundary declaration and responsibility

This is the canonical, single human-facing Atlas dataset-integration entrypoint
for `telco-customer-churn` (Project Spec S0179/S0184). It supersedes the prior
authoring-only stop boundary: this notebook orchestrates Atlas input
verification, external-evidence-authoring, dataset-specific semantic
authoring, capability-aware projection, external fitted-model governed
materialization, inference-bundle materialization, release-candidate
assembly, publisher structural validation, conditional manifest generation,
and one explicit validated-run terminal outcome.

It remains an orchestrator: reusable generic implementation logic lives in
`pipeline/` and `publisher/` modules, never in notebook cells. It stops
unconditionally before publisher promotion, registry activation, or runtime
prediction. The historical `01_dataset_authoring.ipynb` remains read-only
provenance.

In [ ]:
ORCHESTRATION_BOUNDARY = {
    "allowed": [
        "atlas_input_verification",
        "external_evidence_verification_at_authoring_time",
        "dataset_specific_semantic_authoring",
        "capability_profile_declaration_or_reference",
        "capability_aware_projection",
        "external_fitted_model_governed_materialization",
        "inference_bundle_materialization",
        "release_candidate_assembly",
        "publisher_structural_validation",
        "manifest_generation_when_structurally_permitted",
        "validated_run_terminal_outcome",
    ],
    "still_forbidden": [
        "external_eda_rerun",
        "model_fitting_or_retraining",
        "model_selection",
        "threshold_optimization",
        "model_deserialization_or_inference_execution",
        "publisher_promotion",
        "registry_active_release_mutation",
        "public_visibility_or_profile_activation",
    ],
    "external_evidence_is_authoring_time_only": True,
    "external_analysis_is_not_reexecuted": True,
    "durable_absolute_external_root": False,
    "stops_before_promotion_registry_activation_and_runtime_prediction": True,
}
assert ORCHESTRATION_BOUNDARY["durable_absolute_external_root"] is False
assert ORCHESTRATION_BOUNDARY["stops_before_promotion_registry_activation_and_runtime_prediction"] is True
assert set(ORCHESTRATION_BOUNDARY["allowed"]).isdisjoint(ORCHESTRATION_BOUNDARY["still_forbidden"])

## 2. Dataset/source input identity

All durable Atlas inputs and outputs are repository-relative. The Atlas
repository root is resolved through the existing generic resolver
(`pipeline.discovery_evidence.resolve_repository_root`), never assigned as
`Path.cwd().resolve()`, so this notebook resolves Atlas correctly whether the
Jupyter working directory is the repository root, the notebook directory, or
another descendant the resolver can walk up from. The external scientific
project root is an interactive session value and is never copied into an
Atlas artifact or a durable Atlas path.

In [ ]:
from pathlib import Path
import hashlib
import json
import tempfile
from datetime import datetime, timezone

from pipeline.discovery_evidence import resolve_repository_root, resolve_repository_path

repo_root = resolve_repository_root()
dataset_slug = "telco-customer-churn"
dataset_relative_path = "data/raw/telco-customer-churn.csv"

# Session-only input: set by the operator before running Section 4. Never
# persisted into a durable Atlas artifact.
external_scientific_analysis_root = None

external_evidence_index_relative_path = "artifacts/telco-customer-churn/external-evidence-index.json"
capability_profile_relative_path = "pipeline/capabilities/binary-predictive-classification.v1.json"
execution_contract_relative_path = "contracts/telco-customer-churn/execution-contract.json"
runtime_contract_relative_path = "contracts/telco-customer-churn/runtime-contract.json"
public_contract_relative_path = "contracts/telco-customer-churn/public-contract.json"
dataset_context_relative_path = "contracts/telco-customer-churn/dataset-context.json"
discovery_evidence_relative_path = "pipeline/evidence/telco-customer-churn/discovery-evidence.json"
prepared_data_metadata_relative_path = "pipeline/prepared/telco-customer-churn/prepared-data-metadata.json"
authoring_root_relative_path = "pipeline/authoring/telco-customer-churn"
external_fitted_model_run_root_relative_path = "pipeline/external-fitted-model-runs/telco-customer-churn"
authoring_generation_id = "telco-authoring-v1"
canonical_notebook_ref = "notebooks/datasets/telco-customer-churn/dataset_integration.ipynb"
generated_at = "2026-08-11T00:00:00+00:00"

run_state = {"blocked": False, "reasons": []}


def record_block(code_, message, field=None):
    reason = {"code": code_, "message": message}
    if field is not None:
        reason["field"] = field
    run_state["blocked"] = True
    run_state["reasons"].append(reason)
    return reason

## 3. Atlas-owned source verification and drift checks

These observations describe what Atlas sees in the exact current CSV. They
are not imported scientific conclusions.

In [ ]:
from pipeline.discovery_evidence import (
    load_dataset_csv, observe_authoring_fields, resolve_repository_path,
    summarize_structure, summarize_target_column, summarize_identifier_columns,
)
dataset_path = resolve_repository_path(dataset_relative_path, repo_root=repo_root)
rows = load_dataset_csv(dataset_path)
atlas_structure = summarize_structure(rows)
assert atlas_structure["row_count"] == 7043
assert atlas_structure["column_count"] == 21
atlas_field_observations = observe_authoring_fields(rows, atlas_structure["ordered_columns"])
atlas_target_observation = summarize_target_column(rows, "Churn")
atlas_identifier_observation = summarize_identifier_columns(rows, ["customerID"])
assert set(atlas_target_observation["observed_labels"]) == {"No", "Yes"}
assert atlas_identifier_observation[0]["is_unique_per_row"]

## 4. External evidence discovery (`external-evidence-index.v1`)

The real S0165/S0179 producer contract is located beneath an explicitly
supplied external root at
`artifacts/telco-customer-churn/external-evidence-index.json`. Atlas does not
copy external notebooks or rerun that project, and never invents a second
index to satisfy a stale contract.

In [ ]:
external_scientific_analysis_root = (
    "/home/fabyuu/Projetos/DATASET-ANALISYS/"
    "dataset-study-telco-customer-churn"
)

external_root = Path(external_scientific_analysis_root).expanduser().resolve()

assert external_root.is_dir(), external_root

expected_index = (
    external_root
    / "artifacts/telco-customer-churn/external-evidence-index.json"
)

assert expected_index.is_file(), expected_index

print("External root:", external_root)
print("Evidence index:", expected_index)

In [ ]:
def safe_external_relative_path(value):
    candidate = Path(value)
    return bool(value) and not candidate.is_absolute() and ".." not in candidate.parts


assert external_scientific_analysis_root, (
    "Supply the external scientific project root as a session-only input "
    "before running this cell."
)
external_root = Path(external_scientific_analysis_root).expanduser().resolve()
assert safe_external_relative_path(external_evidence_index_relative_path)
external_index_path = external_root / external_evidence_index_relative_path
external_evidence_index = json.loads(external_index_path.read_text(encoding="utf-8"))

assert external_evidence_index.get("schema_version") == "external-evidence-index.v1"
assert external_evidence_index.get("artifact_type") == "external_evidence_index"
assert external_evidence_index.get("dataset_slug") == dataset_slug

provenance = external_evidence_index.get("provenance", {})
assert provenance.get("logical_producer_project_id")
# provenance.repository_revision is provenance context only -- never proof
# that gitignored generated evidence historically belongs to that commit.

evidence_inventory = external_evidence_index.get("evidence_inventory")
assert isinstance(evidence_inventory, list)
evidence_by_role = {item["logical_role"]: item for item in evidence_inventory}

missing_artifacts = external_evidence_index.get("missing_artifacts", [])
required_missing = [item for item in missing_artifacts if item.get("required")]
assert not required_missing, f"required external evidence is missing: {required_missing}"

## 5. External evidence integrity and provenance verification

Every external evidence item actually consumed requires a safe
`relative_path`, resolves only beneath the explicit external root, has its
SHA-256 recomputed from the referenced bytes, and is matched against the
declared `sha256` before any semantic or model use. Only reduced provenance
and relative identities enter durable Atlas artifacts -- never the absolute
external root.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(65536), b""):
            digest.update(block)
    return digest.hexdigest()


def load_verified_external_evidence(logical_role):
    reference = evidence_by_role[logical_role]
    relative_path = reference["relative_path"]
    assert safe_external_relative_path(relative_path)
    evidence_path = external_root / relative_path
    assert sha256_file(evidence_path) == reference["sha256"]
    reduced_reference = {
        "logical_producer_project_id": provenance["logical_producer_project_id"],
        "artifact_type": reference["artifact_type"],
        "artifact_version": reference["artifact_version"],
        "relative_path": relative_path,
        "sha256": reference["sha256"],
        "producer_revision_known": bool(provenance.get("repository_revision")),
        "producer_revision": provenance.get("repository_revision"),
    }
    return json.loads(evidence_path.read_text(encoding="utf-8")), reduced_reference


final_model_manifest, final_model_manifest_ref = load_verified_external_evidence("final_model")
final_test_evidence, final_test_evidence_ref = load_verified_external_evidence("final_test")
model_selection_manifest, model_selection_manifest_ref = load_verified_external_evidence("model_selection")
model_selection_candidates, model_selection_candidates_ref = load_verified_external_evidence("model_selection_candidates")
threshold_analysis, threshold_analysis_ref = load_verified_external_evidence("threshold")
final_model_handoff, final_model_handoff_ref = load_verified_external_evidence("final_model_handoff")
readiness_and_limitations, readiness_and_limitations_ref = load_verified_external_evidence("readiness_and_limitations")

selected_external_evidence = [
    final_model_manifest_ref,
    final_test_evidence_ref,
    model_selection_manifest_ref,
    model_selection_candidates_ref,
    threshold_analysis_ref,
    final_model_handoff_ref,
    readiness_and_limitations_ref,
]
assert all("external_scientific_analysis_root" not in json.dumps(item) for item in selected_external_evidence)

## 6. Dataset-specific semantic interpretation

Telco-specific field meaning, inclusion decisions, missing-value intent,
categorical intent, target semantics, public meaning, and evidence rationale
are materialized in `dataset-semantic-intent.v1`.

In [ ]:
feature_names = [name for name in atlas_structure["ordered_columns"] if name not in {"customerID", "Churn"}]
field_role_decisions = [{"field_name": "customerID", "role": "identifier", "include_in_features": False}]
field_role_decisions += [{
    "field_name": name, "role": "feature", "include_in_features": True,
    "missing_value_intent": ({"policy": "impute_fixed_value", "fixed_value": 0.0, "rationale": "Blank TotalCharges occurs with zero tenure."} if name == "TotalCharges" else {"policy": "no_missing_expected"}),
} for name in feature_names]
field_role_decisions.append({"field_name": "Churn", "role": "target", "include_in_features": False, "exclusion_reason": "Binary result field."})
semantic_intent = {
    "schema_version": "dataset-semantic-intent.v1", "artifact_type": "dataset_semantic_intent",
    "dataset_identity": {"dataset_slug": dataset_slug, "dataset_logical_name": "Telco Customer Churn"},
    "authoring_generation_id": authoring_generation_id,
    "governing_capability_profile": {"capability_profile_id": "binary-predictive-classification", "capability_profile_version": "v1"},
    "field_role_decisions": field_role_decisions,
    "target_semantics": {"target_field_name": "Churn", "task_type": "binary_classification", "positive_class": {"class_id": "Yes", "event_label": "customer churned"}, "is_final_training_configuration": False},
    "authored_public_meaning": {"human_reviewed": True, "safe_projection_intent": "Estimate customer churn propensity from reviewed service and account fields."},
    "authoring_rationale_refs": [{"reference_kind": "source_verification_evidence", "reference_id": "atlas-current-source-observation"}] + [{"reference_kind": "external_scientific_evidence", "reference_id": item["relative_path"]} for item in selected_external_evidence],
    "semantic_boundary_confirmations": {"observed_source_statistics_embedded": False, "scientific_conclusions_embedded": False, "training_outcome_embedded": False, "release_state_embedded": False, "model_bytes_embedded": False},
    "generated_at": generated_at,
}

## 7. Capability-profile resolution

The concrete, generic Atlas-owned capability profile is read and hashed from
`pipeline/capabilities/binary-predictive-classification.v1.json`. It is never
duplicated as dataset-specific content.

In [ ]:
capability_profile_path = repo_root / capability_profile_relative_path
capability_profile = json.loads(capability_profile_path.read_text(encoding="utf-8"))
assert capability_profile["schema_version"] == "capability-profile.v1"
assert capability_profile["capability_profile_id"] == "binary-predictive-classification"
assert capability_profile["capability_profile_version"] == "v1"
assert capability_profile["support_status"] == "current_supported"

## 8. Deterministic preparation/input policy

The governed preparation role records the reviewed, deterministic
`TotalCharges` rule and ordered source fields; it does not perform downstream
modeling work.

In [ ]:
preparation_policy = {
    "schema_version": "candidate-preparation-recipe.v1",
    "dataset_slug": dataset_slug,
    "source_data_ref": dataset_relative_path,
    "ordered_input_columns": atlas_structure["ordered_columns"],
    "transformations": [{"field": "TotalCharges", "operation": "conditional_blank_to_zero", "when": {"field": "tenure", "equals": "0"}, "otherwise": "reject_blank"}],
    "deterministic": True,
}

## 9. Atlas authoring artifact materialization

The semantic intent and preparation policy are durable governed artifacts.
The principal manifest coordinates them and Atlas source verification by
safe relative references and hashes; external provenance contains no
absolute root.

In [ ]:
def write_governed_json(relative_path, payload):
    path = repo_root / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return {"path": relative_path, "sha256": sha256_file(path)}

semantic_ref = write_governed_json(f"{authoring_root_relative_path}/dataset-semantic-intent.json", semantic_intent)
preparation_ref = write_governed_json(f"{authoring_root_relative_path}/preparation-recipe.json", preparation_policy)
source_verification_ref = {"path": discovery_evidence_relative_path, "sha256": sha256_file(repo_root / discovery_evidence_relative_path)}
artifact_references = [
    {"role": "discovery_evidence", **source_verification_ref, "contract_version": "dataset-discovery-evidence.v1"},
    {"role": "semantic_intent", **semantic_ref, "contract_version": "dataset-semantic-intent.v1"},
    {"role": "preparation_recipe", **preparation_ref, "contract_version": "candidate-preparation-recipe.v1"},
]
manifest = {
    "schema_version": "dataset-integration-authoring-manifest.v1", "artifact_type": "dataset_integration_authoring_manifest",
    "dataset_identity": {"dataset_slug": dataset_slug, "dataset_logical_name": "Telco Customer Churn"},
    "authoring_generation": {"authoring_generation_id": authoring_generation_id, "immutable": True, "generated_at": generated_at},
    "capability_profile_selection": {"capability_profile_id": capability_profile["capability_profile_id"], "capability_profile_version": capability_profile["capability_profile_version"], "capability_profile_ref": {"path": capability_profile_relative_path, "sha256": sha256_file(capability_profile_path)}},
    "artifact_references": artifact_references,
    "provenance": [{**item, "artifact_role": "external_scientific_evidence", "input_references": [], "generation_timestamp": None} for item in selected_external_evidence],
    "boundary_confirmations": {"complete_discovery_evidence_embedded": False, "complete_semantic_intent_embedded": False, "complete_preparation_recipe_embedded": False, "training_metrics_embedded": False, "model_selection_payload_embedded": False, "model_bytes_embedded": False, "inference_bundle_payload_embedded": False, "visual_payloads_embedded": False, "absolute_external_project_root_present": False, "external_analysis_handoff_replacement": False, "operational_importer_instruction_present": False},
    "generated_at": generated_at,
}
manifest_ref = write_governed_json(f"{authoring_root_relative_path}/dataset-integration-authoring-manifest.json", manifest)

## 10. Cross-artifact authoring validation

The generic S0166 validator checks schemas, identities, capability
applicability, safe paths, and referenced hashes.

In [ ]:
from pipeline.authoring_contracts import validate_authoring_contracts
authoring_validation = validate_authoring_contracts(manifest, capability_profile, semantic_intent=semantic_intent, artifact_root=repo_root, expected_dataset_slug=dataset_slug, generated_at=generated_at)
assert authoring_validation.valid, authoring_validation.failures

## 11. Capability-aware source/projection handoff

The source input carries the immutable authoring-generation and governed
manifest/profile references, and now references this notebook's own
canonical path. The generic S0167 projector resolves the capability
boundary; the notebook does not implement capability-specific projection
logic.

In [ ]:
from pipeline.contract_derivation import project_capability_aware_source_contract
source_contract_input = {
    "schema_version": "source-contract-input.v1", "dataset_slug": dataset_slug,
    "release_id": authoring_generation_id, "source_contract_ref": runtime_contract_relative_path, "source_data_ref": dataset_relative_path,
    "source_notebook_ref": canonical_notebook_ref,
    "authoring_generation_id": authoring_generation_id, "authoring_manifest_ref": {**manifest_ref, "contract_version": "dataset-integration-authoring-manifest.v1"},
    "capability_profile_id": capability_profile["capability_profile_id"], "capability_profile_version": capability_profile["capability_profile_version"],
    "capability_profile_ref": {"path": capability_profile_relative_path, "sha256": sha256_file(capability_profile_path)},
}
projection_handoff = project_capability_aware_source_contract(source_contract_input, repo_root=repo_root)
assert projection_handoff.authoring_boundary_valid is True

## 12. External fitted-model governed materialization

The verified external evidence is mapped into the existing explicit inputs
required by
`pipeline.materialize_external_fitted_model.materialize_external_fitted_model`.
Model bytes are hash-verified without deserialization; runtime compatibility
comes from the governed manifest, not README prose; the educational
threshold remains educational evidence; operational validity/threshold/
prediction-availability are propagated fail-closed; Atlas output paths are
explicit, repository-relative, and safe; no call reaches
`pipeline.training.train_from_paths`. A `status: blocked` result is carried
into the terminal orchestration outcome rather than treated as a
materialized model.

In [ ]:
from pipeline.materialize_external_fitted_model import materialize_external_fitted_model

run_started_at = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
materialization_id = f"{dataset_slug}-external-materialization-{run_started_at}"
external_fitted_model_run_relative_path = f"{external_fitted_model_run_root_relative_path}/{materialization_id}"

primary_metric = model_selection_manifest["scoring_contract"]["primary_metric"]
selected_candidate = next(
    c for c in model_selection_candidates["candidates"] if c["model_id"] == final_model_manifest["selected_model_id"]
)
selection_choice = model_selection_candidates["selection"]

# Project Spec S0186: deterministic Atlas-native tie-break criteria
# projection. The external producer's raw criteria_applied is never copied
# verbatim into Atlas evidence -- each producer criterion must have an
# explicit canonical mapping, and each first_value/second_value must
# uniquely resolve to a candidate_id from the verified
# model_selection_candidates["candidates"] validation evidence before it is
# trusted.
CANONICAL_TIE_BREAK_CRITERION_MAP = {"lower_validation_brier_score": "brier_score"}


class TieBreakProjectionBlocked(Exception):
    def __init__(self, code_, message):
        self.code = code_
        self.message = message
        super().__init__(message)


def _resolve_candidate_id_by_validation_metric(value, metric_name, candidates):
    matches = [
        c["model_id"]
        for c in candidates
        if c.get("validation_metrics_at_0_50", {}).get(metric_name) == value
    ]
    if len(matches) != 1:
        return None
    return matches[0]


def project_tie_break_criteria(criteria_applied, candidates):
    projected = []
    for order, criterion_entry in enumerate(criteria_applied, start=1):
        producer_criterion = criterion_entry.get("criterion")
        canonical_criterion = CANONICAL_TIE_BREAK_CRITERION_MAP.get(producer_criterion)
        if canonical_criterion is None:
            raise TieBreakProjectionBlocked(
                "unsupported_tie_break_criterion",
                f"No Atlas mapping exists for producer tie-break criterion {producer_criterion!r}.",
            )
        first_value = criterion_entry.get("first_value")
        second_value = criterion_entry.get("second_value")
        winner = criterion_entry.get("winner")
        first_candidate_id = _resolve_candidate_id_by_validation_metric(first_value, canonical_criterion, candidates)
        second_candidate_id = _resolve_candidate_id_by_validation_metric(second_value, canonical_criterion, candidates)
        if (
            first_candidate_id is None
            or second_candidate_id is None
            or first_candidate_id == second_candidate_id
        ):
            raise TieBreakProjectionBlocked(
                "ambiguous_tie_break_candidate_match",
                f"Could not uniquely resolve candidate identities for tie-break criterion {producer_criterion!r}.",
            )
        if winner not in (first_candidate_id, second_candidate_id):
            raise TieBreakProjectionBlocked(
                "tie_break_winner_not_declared_candidate",
                f"Tie-break winner {winner!r} does not resolve to a declared candidate.",
            )
        if producer_criterion == "lower_validation_brier_score":
            lower_candidate_id = first_candidate_id if first_value <= second_value else second_candidate_id
            if winner != lower_candidate_id:
                raise TieBreakProjectionBlocked(
                    "tie_break_winner_inconsistent_with_canonical_observation",
                    f"Tie-break winner {winner!r} is not consistent with the lower canonical "
                    f"{canonical_criterion} observed value.",
                )
        projected.append({
            "order": order,
            "criterion": canonical_criterion,
            "observed_values": [
                {"candidate_id": first_candidate_id, "value": first_value},
                {"candidate_id": second_candidate_id, "value": second_value},
            ],
        })
    return projected


try:
    canonical_tie_break_criteria = project_tie_break_criteria(
        selection_choice["criteria_applied"], model_selection_candidates["candidates"]
    )
    tie_break_projection_blocked_reason = None
except TieBreakProjectionBlocked as exc:
    canonical_tie_break_criteria = None
    tie_break_projection_blocked_reason = {
        "code": exc.code,
        "message": exc.message,
        "field": "practical_tie.tie_break_criteria",
    }

external_training_parameter_record = {
    "schema_version": "training-parameter-record.external-fitted-model.v1",
    "record_kind": "training_parameter_record",
    "origin": "validated_external_fitted_model",
    "producer": provenance["logical_producer_project_id"],
    "handoff_lineage_reference": final_model_handoff_ref["relative_path"],
    "dataset_identity": {"dataset_slug": dataset_slug},
    "selected_model_id": final_model_manifest["selected_model_id"],
    "model_family": final_model_manifest["selected_model_id"],
    "estimator_identity": {"library": "scikit-learn", "class_name": final_model_manifest["selected_model_family"]},
    "hyperparameters": final_model_manifest["selected_hyperparameters"],
    "feature_order": final_model_manifest["feature_columns"],
    "preprocessing_evidence_reference": final_model_manifest_ref["relative_path"],
    "serializer_metadata_reference": final_model_manifest_ref["relative_path"],
    "model_artifact_reference": {
        "path": final_model_manifest["model_artifact_path"],
        "sha256": final_model_manifest["model_artifact_byte_sha256"],
    },
    "model_state_fingerprint": final_model_manifest["final_artifact_fingerprints"]["final-pipeline.joblib"]["semantic_sha256"],
    "atlas_fit_confirmation": {"atlas_fit": False, "atlas_tuned": False, "atlas_recalibrated": False, "atlas_altered": False},
    "raw_partition_confirmation": {"raw_partitions_embedded": False},
}

external_training_metrics = {
    "schema_version": "training-metrics.external-fitted-model.v1",
    "artifact_kind": "training_metrics",
    "created_at": generated_at,
    "evidence_identity": {"model_source_mode": "validated_external_fitted_model", "dataset_slug": dataset_slug},
    "cross_validation_summary": {
        "partition_role": "train",
        "used_for_fitting": True,
        "used_for_model_selection": True,
        "used_for_threshold_selection": False,
        "used_for_adjustment": False,
        "sealed_before_finalization": False,
        "metrics": [{"name": primary_metric, "value": selected_candidate["best_cv_average_precision"]}],
    },
    "validation_evaluation": {
        "partition_role": "validation",
        "used_for_fitting": False,
        "used_for_model_selection": True,
        "used_for_threshold_selection": True,
        "sealed_before_finalization": False,
        "metrics": [{"name": primary_metric, "value": selected_candidate["validation_metrics_at_0_50"][primary_metric]}],
    },
}

if tie_break_projection_blocked_reason is None:
    external_model_selection_evidence = {
        "schema_version": "model-selection-evidence.external-fitted-model.v1",
        "artifact_kind": "model_selection_evidence",
        "created_at": generated_at,
        "evidence_identity": {
            "model_source_mode": "validated_external_fitted_model",
            "producer": provenance["logical_producer_project_id"],
            "handoff_lineage_reference": final_model_handoff_ref["relative_path"],
        },
        "dataset_identity": {"dataset_slug": dataset_slug},
        "selection_protocol": {"protocol_id": "cross_validation_with_practical_tie_break"},
        "selection_policy": {"primary_metric": primary_metric, "ranking_direction": "higher_is_better"},
        "cross_validation_summary": {
            "partition_role": "train",
            "metric_summaries": [{"name": primary_metric, "mean": selected_candidate["cv_average_precision_mean"], "standard_deviation": selected_candidate["cv_average_precision_std"]}],
        },
        "validation_metrics": {
            "partition_role": "validation",
            "metrics": [{"name": primary_metric, "value": selected_candidate["validation_metrics_at_0_50"][primary_metric]}],
        },
        "practical_tie": {
            "tolerance": selection_choice["practical_tie_tolerance"],
            "tie_detected": selection_choice["practical_tie"],
            "tie_break_criteria": canonical_tie_break_criteria,
        },
        "candidates": [{"candidate_id": c["model_id"], "model_family": c["model_id"], "estimator_identity": {"library": "scikit-learn", "class_name": c["family"]}} for c in model_selection_candidates["candidates"]],
        "selected_candidate": {"candidate_id": final_model_manifest["selected_model_id"]},
        "selection_rationale": selection_choice["selection_rationale"],
        "sealed_test_confirmation": {
            "test_partition_role": "test",
            "used_for_model_selection": bool(final_test_evidence["test_used_for_model_selection"]),
            "sealed_before_selection": bool(model_selection_manifest["test_partition_sealed"]),
        },
        "path_references": {"model_selection_evidence_reference": model_selection_manifest_ref["relative_path"]},
        "hashes": {"algorithm": "sha256", "model_state_fingerprint": external_training_parameter_record["model_state_fingerprint"]},
        "evidence_policy": {
            "raw_logs_prohibited": True, "raw_runtime_prohibited": True, "raw_api_payloads_prohibited": True,
            "secrets_prohibited": True, "raw_dataset_embedded": False, "model_bytes_embedded": False,
            "notebook_state_embedded": False, "reduced_and_sanitized": True,
        },
    }

    _staging_dir = Path(tempfile.mkdtemp(prefix="atlas-external-fitted-model-staging-"))

    def _write_staged_json(name, payload):
        path = _staging_dir / name
        path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
        return path

    staged_record_path = _write_staged_json("training-parameter-record.json", external_training_parameter_record)
    staged_metrics_path = _write_staged_json("training-metrics.json", external_training_metrics)
    staged_selection_path = _write_staged_json("model-selection-evidence.json", external_model_selection_evidence)

    external_materialization_result = materialize_external_fitted_model(
        dataset_slug=dataset_slug,
        materialization_id=materialization_id,
        producer="notebooks/datasets/telco-customer-churn/dataset_integration.ipynb",
        source_model_path=external_root / final_model_manifest["model_artifact_path"],
        expected_source_model_sha256=final_model_manifest["model_artifact_byte_sha256"],
        source_training_parameter_record_path=staged_record_path,
        source_training_metrics_path=staged_metrics_path,
        source_model_selection_evidence_path=staged_selection_path,
        expected_evidence_hashes={
            "training_parameter_record": sha256_file(staged_record_path),
            "training_metrics": sha256_file(staged_metrics_path),
            "model_selection_evidence": sha256_file(staged_selection_path),
        },
        runtime_compatibility={
            "serialization_format": "joblib",
            "loader_strategy": "joblib_sklearn_predict",
            "load_safety_confirmed": bool(final_model_manifest["readiness"].get("model_artifact_materialized")),
        },
        educational_threshold={"value": final_model_manifest["educational_threshold"], "scenario": final_model_manifest["threshold_scenario_id"]},
        final_test_completion={
            "used_for_threshold_selection": bool(final_test_evidence["test_used_for_threshold_selection"]),
            "evaluation_count": final_test_evidence["test_probability_evaluation_count"],
        },
        operational_readiness={
            "educational_final_model_complete": bool(final_model_manifest["readiness"].get("educational_final_model_completed")),
            "educational_inference_demo_ready": bool(final_model_manifest["readiness"].get("educational_inference_demo_ready")),
            "operational_validity": final_model_manifest["operational_validity"],
            "operational_threshold": {"status": final_model_manifest["operational_threshold"], "value": None},
            "operational_prediction_available": bool(readiness_and_limitations["readiness"].get("operational_deployment_readiness")),
        },
        output_model_artifact_path=f"{external_fitted_model_run_relative_path}/model.bin",
        output_training_parameter_record_path=f"{external_fitted_model_run_relative_path}/training-parameter-record.json",
        output_training_metrics_path=f"{external_fitted_model_run_relative_path}/training-metrics.json",
        output_model_selection_evidence_path=f"{external_fitted_model_run_relative_path}/model-selection-evidence.json",
        repo_root=repo_root,
    )
else:
    # A blocked tie-break projection must stop before any model-selection
    # evidence is staged or the materializer is invoked -- never persist the
    # raw producer object as a substitute.
    external_materialization_result = {
        "schema_version": "external-fitted-model-materialization.v1",
        "artifact_kind": "external_fitted_model_materialization_result",
        "status": "blocked",
        "created_at": generated_at,
        "blocking_reasons": [tie_break_projection_blocked_reason],
        "boundary_confirmations": {
            "atlas_training_invoked": False,
            "estimator_fit_or_fit_transform_called": False,
            "model_selection_performed": False,
            "threshold_optimized": False,
            "inference_performed": False,
            "model_bytes_imported_or_deserialized": False,
            "training_run_identity_fabricated": False,
            "external_models_created_or_recreated": False,
            "external_project_root_retained": False,
        },
    }

materialization_result_ref = None
if external_materialization_result["status"] == "materialized":
    materialization_result_ref = write_governed_json(
        f"{external_fitted_model_run_relative_path}/materialization-result.json", external_materialization_result
    )
else:
    for reason in external_materialization_result["blocking_reasons"]:
        record_block(reason["code"], reason["message"], reason.get("field"))


## 13. Governed inference-bundle generation

Before calling the bundle producer, the Atlas-owned prepared dataset is
resolved and verified from `prepared-data-metadata.json` via the same
governed `pipeline.training._prepared_dataset_metadata_blocking_reasons`
boundary the internal training branch already uses -- never the raw
dataset, never a fabricated path. On successful external materialization
and prepared-dataset resolution, the governed generic bundle producer is
called with the external materialization result, the resolved prepared
dataset reference, and the current Atlas contract/context references.
`model_source_mode` provenance, `external_model_evidence`, and operational
readiness values are preserved verbatim; no `training_evidence` is
fabricated for the external fitted model. A blocked bundle result fails
closed and becomes part of the terminal orchestration reason instead of
being bypassed.

In [ ]:
# Project Spec: prepared-dataset / inference-bundle blocker fix.
#
# The external inference-bundle branch requires an Atlas-owned prepared
# dataset (contracts/inference-bundle.schema.json's `prepared_dataset` is
# unconditionally required). This resolves it the same governed way
# `pipeline.training.materialize_training_run_from_prepared_metadata`
# already does for the internal branch -- via
# `pipeline.training._prepared_dataset_metadata_blocking_reasons` against
# the Atlas-owned `prepared-data-metadata.json` -- never the raw dataset,
# never a fabricated/guessed path.
from pipeline.training import _prepared_dataset_metadata_blocking_reasons

prepared_dataset_path = None
prepared_dataset_ref = None

if not run_state["blocked"]:
    prepared_dataset_blocking_reasons: list[str] = []
    prepared_dataset_metadata_full_path = repo_root / prepared_data_metadata_relative_path
    try:
        prepared_dataset_metadata = json.loads(
            prepared_dataset_metadata_full_path.read_text(encoding="utf-8")
        )
    except (OSError, json.JSONDecodeError) as exc:
        prepared_dataset_metadata = None
        prepared_dataset_blocking_reasons.append(
            f"prepared_data_metadata_path could not be read/parsed: {exc}"
        )

    if prepared_dataset_metadata is not None:
        metadata_dataset_slug = (prepared_dataset_metadata.get("dataset_identity") or {}).get(
            "dataset_slug"
        )
        if metadata_dataset_slug != dataset_slug:
            prepared_dataset_blocking_reasons.append(
                "prepared_data_metadata.dataset_identity.dataset_slug does not match the "
                f"governed dataset_slug: {metadata_dataset_slug!r} != {dataset_slug!r}."
            )
        governed_reasons, governed_reference = _prepared_dataset_metadata_blocking_reasons(
            prepared_dataset_metadata
        )
        prepared_dataset_blocking_reasons.extend(governed_reasons)
        if not prepared_dataset_blocking_reasons and governed_reference:
            prepared_dataset_ref = governed_reference
            prepared_dataset_path = repo_root / governed_reference

    for reason in prepared_dataset_blocking_reasons:
        record_block("prepared_dataset_unresolved", reason)


In [ ]:
from pipeline import generate_inference_bundle

inference_bundle_result = None
if not run_state["blocked"]:
    inference_bundle_output_relative_path = f"{external_fitted_model_run_relative_path}/inference-bundle.json"
    provisional_release_id = f"release-{run_started_at[:8]}-001"
    inference_bundle_result = generate_inference_bundle.materialize_governed_inference_bundle(
        external_fitted_model_materialization_result=external_materialization_result,
        execution_contract_path=repo_root / execution_contract_relative_path,
        runtime_contract_path=repo_root / runtime_contract_relative_path,
        public_contract_path=repo_root / public_contract_relative_path,
        dataset_context_path=repo_root / dataset_context_relative_path,
        prepared_dataset_path=prepared_dataset_path,
        prepared_dataset_ref=prepared_dataset_ref,
        output_path=repo_root / inference_bundle_output_relative_path,
        prediction_type="number",
        repo_root=repo_root,
        dataset_slug=dataset_slug,
        release_id=provisional_release_id,
        class_labels=final_model_manifest["target_classes"],
        probability_output=True,
        execution_contract_ref=execution_contract_relative_path,
        runtime_contract_ref=runtime_contract_relative_path,
        public_contract_ref=public_contract_relative_path,
        dataset_context_ref=dataset_context_relative_path,
    )
    if inference_bundle_result["status"] != "generated":
        for reason in inference_bundle_result["blocking_reasons"]:
            record_block("inference_bundle_blocked", reason)


## 14. Release-candidate assembly from compatible governed roles

Uses the existing generic candidate primitives
(`pipeline.assemble_candidate.build_release_candidate_input`,
`pipeline.assemble_candidate.assemble_release_candidate`), never the legacy
Telco-specific publisher convenience helper. Handoff readiness is checked
first through the existing generic readiness helper; a missing or
incompatible required role (for example, no governed `model_card`,
`public_context`, or `visualizations` artifact yet exists for this external
model) blocks candidate assembly instead of borrowing an unrelated
historical internal-training artifact or fabricating a placeholder.

In [ ]:
from pipeline import assemble_candidate

release_candidate_assembly_result = None
if not run_state["blocked"]:
    candidate_release_id = provisional_release_id
    candidate_artifact_references = {
        "discovery_evidence": discovery_evidence_relative_path,
        "execution_contract": execution_contract_relative_path,
        "runtime_contract": runtime_contract_relative_path,
        "public_contract": public_contract_relative_path,
        "preparation_recipe": preparation_ref["path"],
        "prepared_data_metadata": prepared_data_metadata_relative_path,
        "training_parameter_record": external_materialization_result["evidence_references"]["training_parameter_record_path"],
        "model_artifact": external_materialization_result["model_artifact_path"],
        "training_metrics": external_materialization_result["evidence_references"]["training_metrics_path"],
        # No governed model_card/public_context/visualizations artifact
        # exists yet for this external fitted model -- these roles are left
        # unresolved rather than borrowed from historical internal-training
        # releases or fabricated, so handoff readiness below fails closed.
        "model_card": "",
        "public_context": "",
        "visualizations": "",
        "inference_bundle": inference_bundle_output_relative_path,
    }
    candidate_handoff_readiness = assemble_candidate.build_release_candidate_handoff_readiness(
        candidate_artifact_references, repo_root=repo_root
    )
    if candidate_handoff_readiness["is_release_candidate_input_ready"]:
        candidate_input = assemble_candidate.build_release_candidate_input(
            dataset_slug=dataset_slug,
            release_id=candidate_release_id,
            source_run_id=materialization_id,
            artifact_references=candidate_artifact_references,
            repo_root=repo_root,
        )
        release_candidate_assembly_result = assemble_candidate.assemble_release_candidate(
            candidate_input,
            repo_root / "releases" / "candidates",
            repo_root=repo_root,
        )
        if release_candidate_assembly_result.get("status") != "accepted":
            record_block(
                "candidate_assembly_rejected",
                f"release-candidate assembly did not reach accepted status: {release_candidate_assembly_result.get('reason')}",
            )
    else:
        record_block(
            "candidate_role_unavailable",
            "release-candidate handoff readiness failed for one or more required roles.",
            candidate_handoff_readiness["not_ready_roles"],
        )

## 15. Publisher structural validation and conditional manifest generation

`publisher.validate.run` is called only once candidate assembly succeeds,
using the concrete candidate directory the assembly step returned. The
created run directory is resolved deterministically from the actual
filesystem result of the call -- never guessed from a wall-clock string.
`publisher.manifest.run` is then called for that same run whenever current
generic structural rules permit it (`validation_outcome == "accepted"`),
never as a proxy for promotion eligibility. This never calls
`publisher.validate.materialize_telco_validation_run`, the dataset-specific
legacy convenience path.

In [ ]:
import sys
import importlib.util

repo_root_str = str(repo_root)

if repo_root_str not in sys.path:
    sys.path.insert(0, repo_root_str)

assert importlib.util.find_spec("publisher.validate") is not None
assert importlib.util.find_spec("publisher.manifest") is not None

print("Atlas repo root added to sys.path:", repo_root_str)
print("publisher.validate: OK")
print("publisher.manifest: OK")

In [ ]:
# Structural validation is publisher.validate.run; conditional manifest
# generation is publisher.manifest.run -- imported below with short
# aliases for readability in this cell.
from publisher import validate as publisher_validate
from publisher import manifest as publisher_manifest

publisher_validation_result = None
publisher_manifest_result = None
publisher_run_dir_relative_path = None
publisher_manifest_relative_path = None

if not run_state["blocked"] and release_candidate_assembly_result is not None:
    runs_root = repo_root / "publisher" / "runs"
    existing_run_dirs = {p.name for p in runs_root.iterdir() if p.is_dir()} if runs_root.is_dir() else set()

    publisher_validation_result = publisher_validate.run(
        release_candidate_assembly_result["candidate_dir"], repo_root=repo_root
    )

    new_run_dirs = sorted(p.name for p in runs_root.iterdir() if p.is_dir() and p.name not in existing_run_dirs)
    assert len(new_run_dirs) == 1, "publisher.validate.run must create exactly one new run directory"
    publisher_run_id = new_run_dirs[0]
    publisher_run_dir = runs_root / publisher_run_id
    assert (publisher_run_dir / "validation-result.json").is_file()
    publisher_run_dir_relative_path = str(publisher_run_dir.relative_to(repo_root))

    if publisher_validation_result.get("validation_outcome") == "accepted":
        publisher_manifest_result = publisher_manifest.run(str(publisher_run_dir), repo_root=repo_root)
        manifest_candidate = publisher_run_dir / "manifest.json"
        if manifest_candidate.is_file():
            publisher_manifest_relative_path = str(manifest_candidate.relative_to(repo_root))
    else:
        record_block(
            "publisher_structural_validation_rejected",
            "publisher.validate.run did not accept the assembled candidate.",
        )

## 16. Validated-run terminal result

Exactly one explicit validated-run terminal outcome is materialized through
`pipeline.validated_run.materialize_validated_run_terminal_result`, from
explicit durable references/hashes only. This notebook never reimplements
promotion-eligibility logic; the generic terminal producer owns the
eligibility/hash/schema decisions. A completed run with unresolved,
unconfirmed, or false operational readiness legitimately remains
`promotion_eligibility: false`. Blocked lower stages propagate their concrete
reason codes/messages into this terminal outcome. The schema-valid result is
persisted through a narrow JSON-write step into the same already-governed
publisher run directory used above (or, when no publisher run exists yet,
alongside the external materialization outputs) -- never a newly invented
broad directory convention.

In [ ]:
from pipeline import validated_run


def _durable_ref(path_value):
    if path_value is None:
        return None
    return {"path": path_value, "sha256": sha256_file(repo_root / path_value)}


durable_references = {
    "materialization_result": _durable_ref(materialization_result_ref["path"] if materialization_result_ref else None),
    "inference_bundle": _durable_ref(inference_bundle_output_relative_path if inference_bundle_result and inference_bundle_result.get("status") == "generated" else None),
    "release_candidate": _durable_ref(
        f"{release_candidate_assembly_result['candidate_dir']}/release-candidate.json".replace(str(repo_root) + "/", "")
        if release_candidate_assembly_result and release_candidate_assembly_result.get("status") == "accepted"
        else None
    ),
    "publisher_validation_result": _durable_ref(f"{publisher_run_dir_relative_path}/validation-result.json" if publisher_run_dir_relative_path else None),
    "manifest": _durable_ref(publisher_manifest_relative_path),
    "operational_readiness_source": _durable_ref(materialization_result_ref["path"] if materialization_result_ref else None),
}

structural_validation = None
if publisher_validation_result is not None:
    structural_validation = {"validation_outcome": publisher_validation_result.get("validation_outcome")}

manifest_outcome = None
if publisher_validation_result is not None:
    manifest_outcome = {
        "manifest_generated": publisher_manifest_relative_path is not None,
        "manifest_path": publisher_manifest_relative_path,
    }

if external_materialization_result["status"] == "materialized":
    operational_readiness_source_values = external_materialization_result["operational_readiness"]
    operational_readiness = {
        "operational_validity": operational_readiness_source_values["operational_validity"],
        "operational_threshold": {
            "status": operational_readiness_source_values["operational_threshold"] if isinstance(operational_readiness_source_values["operational_threshold"], str) else operational_readiness_source_values["operational_threshold"].get("status"),
            "value": None,
        },
        "operational_prediction_available": bool(operational_readiness_source_values["operational_prediction_available"]),
    }
else:
    operational_readiness = {
        "operational_validity": "unconfirmed",
        "operational_threshold": {"status": "unresolved", "value": None},
        "operational_prediction_available": False,
    }

terminal_status = "blocked" if run_state["blocked"] else "completed"
terminal_reasons = run_state["reasons"] if run_state["blocked"] else None

validated_run_terminal_result = validated_run.materialize_validated_run_terminal_result(
    run_id=materialization_id,
    dataset_slug=dataset_slug,
    model_source_mode="validated_external_fitted_model",
    status=terminal_status,
    durable_references=durable_references,
    structural_validation=structural_validation,
    manifest_outcome=manifest_outcome,
    operational_readiness=operational_readiness,
    reasons=terminal_reasons,
    repo_root=repo_root,
)

# Persist the schema-valid terminal result narrowly. pipeline.validated_run
# owns every eligibility/hash/schema decision above -- this notebook only
# writes the already-returned object to a safe, repository-relative,
# already-governed run directory.
if publisher_run_dir_relative_path is not None:
    terminal_result_relative_path = f"{publisher_run_dir_relative_path}/validated-run-terminal-result.json"
else:
    terminal_result_relative_path = f"{external_fitted_model_run_relative_path}/validated-run-terminal-result.json"
terminal_result_ref = write_governed_json(terminal_result_relative_path, validated_run_terminal_result)

assert validated_run_terminal_result["status"] in ("completed", "blocked", "failed")
assert validated_run_terminal_result["promotion_eligibility"] in (True, False)

## 17. Orchestration stop confirmation

This notebook's terminal artifact is `validated_run_terminal_result`. It
contains no active call that can run `publisher.promote.run`, mutate
`registry/datasets.json` `active_release`, activate a release, change public
visibility, write profile publication state, load or deserialize the fitted
model, or serve/predict through runtime inference.
`promotion_eligibility: true`, if ever produced by a future governed
readiness state, is informational only inside this notebook -- promotion
remains a separate, operator-controlled work package.

In [ ]:
orchestration_summary = {
    "dataset_slug": dataset_slug,
    "materialization_id": materialization_id,
    "canonical_notebook_ref": canonical_notebook_ref,
    "terminal_status": validated_run_terminal_result["status"],
    "promotion_eligibility": validated_run_terminal_result["promotion_eligibility"],
    "terminal_result_ref": terminal_result_ref,
    "stops_before_promotion_registry_activation_and_runtime_prediction": True,
}
orchestration_summary